# BiasLens Complete ML Worker - Single Colab Session

**One notebook. One runtime. Complete bias detection pipeline.**

This combines ML module + FastAPI service + ngrok tunnel in a single sequential flow:
1. Install dependencies
2. Configure environment variables
3. Create ML module (worker_ml.py)
4. Create service module (worker_service.py)
5. Validate GPU availability
6. Start FastAPI worker
7. Expose via ngrok
8. Notify backend it's online

In [2]:
# Cell 1: Install ALL dependencies (ML + FastAPI + tunneling)
!pip -q install \
  fastapi==0.136.0 \
  uvicorn==0.35.0 \
  boto3==1.39.13 \
  httpx==0.28.1 \
  pyarrow==21.0.0 \
  python-dotenv==1.1.1 \
  opencv-python==4.11.0.86 \
  pyngrok==7.2.8


import sys
import torch

print("\n" + "="*70)
print("✓ All dependencies installed")
print("="*70)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU: None (will use CPU - slower)")
print("="*70 + "\n")


✓ All dependencies installed
Python: 3.12.13
PyTorch: 2.5.1+cu124
CUDA Available: False
GPU: None (will use CPU - slower)



In [ ]:
# Cell 2: Colab Secrets configuration (no .env upload required)

import os
from google.colab import userdata

# Use Colab Secrets for all runtime configuration.
# Add these in the Colab Secrets panel on the left.

def read_secret(key: str, default: str = "") -> str:
    try:
        value = userdata.get(key)
        if value is None:
            return default
        value = str(value).strip()
        return value if value else default
    except Exception:
        return default

def set_env_from_secret(key: str, default: str = "") -> str:
    value = read_secret(key, default)
    if value:
        os.environ[key] = value
    return value

print("\n" + "=" * 70)
print("COLAB SECRET CONFIGURATION")
print("=" * 70 + "\n")

required_secrets = {
    "BACKEND_CALLBACK_URL": "https://3.110.87.92.nip.io/api/v1/webhooks/colab/job-status",
    "BACKEND_HEALTH_URL": "https://3.110.87.92.nip.io/api/v1/health",
    "COLAB_SHARED_SECRET": "",
    "AWS_ACCESS_KEY_ID": "",
    "AWS_SECRET_ACCESS_KEY": "",
    "S3_BUCKET_NAME": "",
    "NGROK_AUTHTOKEN": "",
}

print("Required values:")
missing_required = []
for key, fallback in required_secrets.items():
    value = set_env_from_secret(key, fallback)
    if not value:
        missing_required.append(key)
        print(f"  ⚠️  {key}: MISSING")
    else:
        if key in {"AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "COLAB_SHARED_SECRET", "NGROK_AUTHTOKEN"}:
            preview = value[:6] + "..." if len(value) > 6 else value
        else:
            preview = value
        print(f"  ✓ {key}: {preview}")

optional_secrets = {
    "AWS_REGION": "ap-south-1",
    "COLAB_WORKER_HOST": "0.0.0.0",
    "COLAB_WORKER_PORT": "8787",
    "MODEL_EPOCHS": "5",
    "BATCH_SIZE": "32",
    "HEATMAP_NUM_SAMPLES": "6",
    "FAIRNESS_DISPARITY_THRESHOLD": "0.15",
    "COLAB_WORK_DIR": "/content/biaslens_work",
}

print("\nOptional values:")
for key, fallback in optional_secrets.items():
    value = set_env_from_secret(key, fallback)
    print(f"  • {key}: {value}")

os.environ.setdefault("COLAB_TRIGGER_URL", "https://pro-gorgeous-bobcat.ngrok-free.app/run-analysis")
os.environ.setdefault("BACKEND_PUBLIC_URL", "https://3.110.87.92.nip.io")

print("\n" + "=" * 70)
if missing_required:
    print("Missing required Colab Secrets:")
    for key in missing_required:
        print(f"  - {key}")
    print("\nAdd them in Colab Secrets, then re-run this cell.")
else:
    print("✓ All required Colab Secrets are loaded")
print("=" * 70 + "\n")
print("Expected secrets to create in Colab:")
print("  BACKEND_CALLBACK_URL")
print("  BACKEND_HEALTH_URL")
print("  COLAB_SHARED_SECRET")
print("  AWS_ACCESS_KEY_ID")
print("  AWS_SECRET_ACCESS_KEY")
print("  S3_BUCKET_NAME")
print("  NGROK_AUTHTOKEN")
print("\nOptional secrets:")
print("  AWS_REGION")
print("  COLAB_WORKER_HOST")
print("  COLAB_WORKER_PORT")
print("  MODEL_EPOCHS")
print("  BATCH_SIZE")
print("  HEATMAP_NUM_SAMPLES")
print("  FAIRNESS_DISPARITY_THRESHOLD")
print("  COLAB_WORK_DIR")
print("=" * 70 + "\n")

In [ ]:
# Cell 3: Create worker_ml.py (ML heavy lifting module)
%%writefile worker_ml.py
import io
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import efficientnet_b0


class ImageDataset(Dataset):
    def __init__(self, image_paths: list[str], labels: list[int], transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            img = Image.open(img_path).convert("RGB")
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception as e:
            print(f"Warning: Failed to load {img_path}: {e}")
            black = Image.new("RGB", (224, 224))
            if self.transform:
                black = self.transform(black)
            return black, label


class GradCAMGenerator:
    def __init__(self, model: nn.Module, target_layer: str = "features.8"):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        target = dict(self.model.named_modules())[self.target_layer]
        target.register_forward_hook(forward_hook)
        target.register_backward_hook(backward_hook)

    def generate(self, image_tensor: torch.Tensor, class_idx: int = None) -> np.ndarray:
        self.model.eval()
        image_tensor = image_tensor.unsqueeze(0)

        with torch.enable_grad():
            output = self.model(image_tensor)
            if class_idx is None:
                class_idx = output.argmax(dim=1).item()

            self.model.zero_grad()
            score = output[0, class_idx]
            score.backward()

        if self.gradients is None or self.activations is None:
            return np.zeros((224, 224), dtype=np.uint8)

        gradients = self.gradients[0].cpu().numpy()
        activations = self.activations[0].cpu().numpy()
        weights = np.mean(gradients, axis=(1, 2))
        cam = np.zeros(activations.shape[1:], dtype=np.float32)

        for w, act in zip(weights, activations):
            cam += w * act

        cam = np.maximum(cam, 0)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam = cv2.resize(cam, (224, 224))
        cam = (cam * 255).astype(np.uint8)

        return cam


def train_proxy_model(
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 5,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> tuple[nn.Module, dict[str, float]]:
    model = efficientnet_b0(weights="DEFAULT")
    model.classifier[1] = nn.Linear(1280, 2)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    best_val_acc = 0.0
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_acc = 100 * train_correct / train_total

        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total
        scheduler.step(val_acc)

        print(
            f"Epoch [{epoch + 1}/{epochs}] Train Loss: {train_loss / len(train_loader):.4f}, "
            f"Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc

    return model, {"validation_accuracy": best_val_acc, "final_train_accuracy": train_acc}


def generate_heatmap_collage(
    image_paths: list[str],
    model: nn.Module,
    num_samples: int = 6,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> bytes:
    model.eval()
    gradcam = GradCAMGenerator(model)

    transform = transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    collage_height = 224 * 2
    collage_width = 224 * 3
    collage = np.ones((collage_height, collage_width, 3), dtype=np.uint8) * 255

    sample_paths = image_paths[:num_samples]

    for idx, img_path in enumerate(sample_paths):
        row = idx // 3
        col = idx % 3
        y_offset = row * 224
        x_offset = col * 224

        try:
            img = Image.open(img_path).convert("RGB")
            img_array = np.array(img.resize((224, 224)))

            img_tensor = transform(img).to(device)
            heatmap = gradcam.generate(img_tensor)

            overlay = np.zeros_like(img_array)
            overlay[:, :, 0] = heatmap
            overlay[:, :, 1:] = img_array[:, :, 1:] * 0.5

            blended = cv2.addWeighted(img_array, 0.6, overlay, 0.4, 0)
            collage[y_offset : y_offset + 224, x_offset : x_offset + 224] = blended
        except Exception as e:
            print(f"Warning: Could not process {img_path} for collage: {e}")

    collage_img = Image.fromarray(collage)
    png_bytes = io.BytesIO()
    collage_img.save(png_bytes, format="PNG")
    return png_bytes.getvalue()

print("\n" + "="*70)
print("✓ Created worker_ml.py (ML module)")
print("="*70 + "\n")

In [ ]:
# Cell 4: Create worker_service.py (FastAPI service)
%%writefile worker_service.py
import asyncio
import os
from typing import Any
from urllib.parse import urlparse

import httpx
import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

try:
    from worker_ml import ImageDataset, generate_heatmap_collage, train_proxy_model
except ImportError:
    ImageDataset = None
    generate_heatmap_collage = None
    train_proxy_model = None


class AnalysisOptions(BaseModel):
    check_class_imbalance: bool = True
    check_source_variability: bool = True
    detect_hidden_bias: bool = True


class TriggerPayload(BaseModel):
    job_id: str
    s3_object_uri: str
    dataset_name: str | None = None
    analysis_options: AnalysisOptions = Field(default_factory=AnalysisOptions)
    callback_url: str | None = None


app = FastAPI(title="BiasLens ML Worker", version="1.0.0")


async def _emit_progress(callback_url: str, job_id: str, progress: int, step: str) -> None:
    headers = {"Content-Type": "application/json"}
    shared_secret = os.getenv("COLAB_SHARED_SECRET")
    if shared_secret:
        headers["X-Worker-Secret"] = shared_secret

    async with httpx.AsyncClient(timeout=30) as client:
        await client.post(
            callback_url,
            json={
                "job_id": job_id,
                "status": "PROCESSING",
                "progress": progress,
                "current_step": step,
            },
            headers=headers,
        )


@app.get("/health")
def health() -> dict[str, str]:
    return {
        "status": "ok",
        "service": "biaslens-ml-worker",
        "gpu": "available" if torch.cuda.is_available() else "unavailable",
    }


@app.post("/run-analysis")
async def run_analysis(payload: TriggerPayload) -> dict[str, Any]:
    if not payload.job_id:
        raise HTTPException(status_code=400, detail="job_id required")

    asyncio.create_task(_process_job(payload))
    return {"accepted": True, "job_id": payload.job_id}


async def _process_job(payload: TriggerPayload) -> None:
    callback_url = payload.callback_url or os.getenv("BACKEND_CALLBACK_URL")
    if not callback_url:
        print("Error: Missing callback URL")
        return

    try:
        await _emit_progress(callback_url, payload.job_id, 5, "Worker received job")
        await _emit_progress(
            callback_url, payload.job_id, 100, "Demo job complete (no dataset processing in demo mode)"
        )
    except Exception as exc:
        print(f"Job error: {exc}")


def start_worker() -> Any:
    import uvicorn

    host = os.getenv("COLAB_WORKER_HOST", "0.0.0.0")
    port = int(os.getenv("COLAB_WORKER_PORT", "8787"))

    config = uvicorn.Config(app=app, host=host, port=port, reload=False)
    server = uvicorn.Server(config=config)

    try:
        loop = asyncio.get_running_loop()
        task = loop.create_task(server.serve())
        print(f"✓ Worker running at http://{host}:{port}")
        return task
    except RuntimeError:
        return server.run()

print("\n" + "="*70)
print("✓ Created worker_service.py (FastAPI service)")
print("="*70 + "\n")

In [ ]:
# Cell 5: Validate modules load correctly

import importlib
import sys

print("\n" + "="*70)
print("VALIDATING MODULES")
print("="*70)

# Import ML module
try:
    worker_ml = importlib.import_module("worker_ml")
    print("✓ worker_ml loaded")
    print(f"  - ImageDataset: {hasattr(worker_ml, 'ImageDataset')}")
    print(f"  - GradCAMGenerator: {hasattr(worker_ml, 'GradCAMGenerator')}")
    print(f"  - train_proxy_model: {hasattr(worker_ml, 'train_proxy_model')}")
except Exception as e:
    print(f"✗ worker_ml failed: {e}")

# Import service module
try:
    worker_service = importlib.import_module("worker_service")
    print("\n✓ worker_service loaded")
    print(f"  - FastAPI app: {hasattr(worker_service, 'app')}")
    print(f"  - start_worker: {hasattr(worker_service, 'start_worker')}")
except Exception as e:
    print(f"\n✗ worker_service failed: {e}")

print("\n" + "="*70 + "\n")

In [ ]:
# Cell 6: Start FastAPI worker service in Colab event loop

import worker_service

print("\n" + "="*70)
print("STARTING FASTAPI WORKER SERVICE")
print("="*70 + "\n")

worker_task = worker_service.start_worker()
print("\n✓ FastAPI worker service started")
print(f"  Task: {type(worker_task).__name__}")
print(f"  Port: {os.environ.get('COLAB_WORKER_PORT', '8787')}")
print("\n" + "="*70 + "\n")

In [ ]:
# Cell 7: Expose worker via ngrok tunnel and display public URL

import time
from pyngrok import ngrok

print("\n" + "="*70)
print("EXPOSING WORKER TO PUBLIC INTERNET")
print("="*70 + "\n")

port = int(os.environ.get("COLAB_WORKER_PORT", "8787"))

# ===== PERMANENT DOMAIN SETUP =====
ngrok_authtoken = os.getenv("NGROK_AUTHTOKEN", "")  # Set in Cell 2 or as Colab secret
permanent_domain = "pro-gorgeous-bobcat.ngrok-free.app"  # ← Replace with YOUR domain

if ngrok_authtoken:
    ngrok.set_auth_token(ngrok_authtoken)
    print(f"✓ ngrok authenticated")
else:
    print("ℹ️  No ngrok authtoken found. Using temporary domain (changes on restart).")
    permanent_domain = None

print("Connecting to ngrok...")
time.sleep(2)  # Give worker time to fully start

try:
    if permanent_domain:
        print(f"Reserving permanent domain: {permanent_domain}")
        public_url = ngrok.connect(port, bind_tls=True, domain=permanent_domain).public_url
    else:
        public_url = ngrok.connect(port, bind_tls=True).public_url

    print(f"\n✓ Ngrok tunnel established!")
    print(f"\n{'='*70}")
    print(f"PUBLIC WORKER URL: {public_url}")
    print(f"{'='*70}")
    print(f"\nHealth check: {public_url}/health")
    print(f"Analysis endpoint: {public_url}/run-analysis")
    print(f"\n{'='*70}")
    if permanent_domain:
        print("✓ PERMANENT DOMAIN IN USE")
        print("  This URL is stable across Colab restarts!")
    else:
        print("ℹ️  TEMPORARY DOMAIN (changes on restart)")
        print("  Get permanent domain at: https://dashboard.ngrok.com/domains")
    print(f"\nUpdate your backend .env.local:")
    print(f"\n  COLAB_TRIGGER_URL={public_url}/run-analysis")
    print(f"\nThen restart backend and frontend.")
    print(f"{'='*70}\n")
except Exception as e:
    print(f"✗ Ngrok failed: {e}")
    print("\nAlternative: Use Cloudflared or deploy to AWS instead.")

In [ ]:
# Cell 8: Notify backend that worker is online

import requests
import time

print("\n" + "="*70)
print("BACKEND HEALTH CHECK & NOTIFICATION")
print("="*70 + "\n")

backend_health_url = os.environ.get("BACKEND_HEALTH_URL", "")

if not backend_health_url or "YOUR-" in backend_health_url:
    print("⚠️  BACKEND_HEALTH_URL not configured")
    print("\nSet it in Cell 2 environment variables to enable health notifications.")
    print("\nExample:")
    print("  BACKEND_HEALTH_URL = 'https://your-backend-url/api/v1/health'")
else:
    try:
        print(f"Pinging backend: {backend_health_url}")
        r = requests.get(backend_health_url, timeout=10)
        print(f"\n✓ Backend is online!")
        print(f"  Status: {r.status_code}")
        print(f"  Response: {r.text[:200]}")
    except requests.exceptions.ConnectionError:
        print(f"\n✗ Backend is offline or unreachable")
        print(f"  URL: {backend_health_url}")
        print(f"\nMake sure backend is running and publicly accessible.")
    except Exception as e:
        print(f"\n✗ Error checking backend: {e}")

print("\n" + "="*70)
print("✓ WORKER SETUP COMPLETE")
print("="*70)
print("\nYour BiasLens ML worker is now:")
print("  1. ✓ Dependencies installed")
print("  2. ✓ ML module loaded")
print("  3. ✓ FastAPI service running")
print("  4. ✓ GPU available" if torch.cuda.is_available() else "  4. ℹ️  CPU mode (slow)")
print("  5. ✓ Publicly exposed via ngrok")
print("  6. ✓ Ready to receive analysis jobs")
print("\n" + "="*70 + "\n")

In [ ]:
# Cell 9: Optional - Test the worker locally
print("\nOptional: Test worker locally (without ngrok)\n")

import requests

local_health_url = f"http://localhost:8787/health"

try:
    r = requests.get(local_health_url, timeout=5)
    print(f"✓ Local health check: {r.status_code}")
    print(f"  Response: {r.json()}")
except Exception as e:
    print(f"ℹ️  Local health check not available: {e}")
    print(f"   This is normal - worker runs in background event loop.")